# Zpracování dat pro ArcGIS Online

Notebook převádí data monitoringu (Excel sešity) na GeoJSON vrstvy
použitelné v ArcGIS Online a dále provádí analýzu časového vývoje aktivity
$^{137}$Cs v jednotlivých typech vzorků.

## 1. Potřebné knihovny

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
import geopandas as gpd
import matplotlib.pyplot as plt
import yaml

## 2. Nastavitelné proměnné

In [ ]:
# Všechny roky měření (jako string, protože odpovídají názvům listů v Excelu)
Rok = {
    "2000", "2001", "2002", "2003", "2004", "2005", "2006", "2007",
    "2008", "2009", "2010", "2011", "2012", "2013", "2014", "2015",
    "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024",
}

# Soubor se souřadnicemi odběrových bodů
dataframe_souradnice = pd.read_excel("Body_lokace.xlsx", sheet_name="Body", decimal=",")

# Soubor s naměřenými daty (listy ETE_<rok>, ETE_<rok>_In-situ, ETE_<rok>_GT40)
Soubor = "ETEDATA.xlsx"

# Přiřazení druhu vzorku ke zkratce použité v názvu vzorku (např. "S12" -> mech, bod 12)
typ_vzorku = {
    "S": "Mech trávník Schreberův (Pleurozium schreberi)",
    "H": "Hřib hnědý (Xerocomus badius)",
    "K": "Vnější kůra borovice lesní (Pinus sylvestris)",
    "LH": "Lesní nadložní humus",
    "HK": "Koš hub (mix jedlých hub)",
    "B": "Brusnice borůvka (Vaccinium myrtillus)",
    "M": "Malina (Rubus idaeus)",
    "O": "Ostružina (Rubus fructicosus)",
    "R": "Hřib hořký (Boletus radicans)",
}

# Rozlišení horizontu humusu (poslední písmeno v názvu vzorku, např. "LH12A")
typ_humusu = {
    "A": "nehumifikovaný (litter)",
    "B": "humifikovaný (fermenton)",
}

# Anglické ekvivalenty výše (pro dvojjazyčné popupy v ArcGIS/StoryMaps)
typ_vzorku_en = {
    "S": "Schreber's Moss (Pleurozium schreberi)",
    "H": "Bay Bolete (Xerocomus badius)",
    "K": "Outer Bark of Scots Pine (Pinus sylvestris)",
    "LH": "Forest Floor Humus",
    "HK": "Mushroom Basket (mix of edible mushrooms)",
    "B": "Bilberry (Vaccinium myrtillus)",
    "M": "Raspberry (Rubus idaeus)",
    "O": "Blackberry (Rubus fructicosus)",
    "R": "Bitter Bolete (Boletus radicans)",
}

typ_humusu_en = {
    "A": "non-humified (litter)",
    "B": "humified (fermentation layer)",
}

# Přiřazení měřené veličiny pro data z přístroje GT-40 (CZ / EN)
mapping_cz = {
    "DAVKOVY_PRIKON": "Dávkový příkon",
    "KONCENTRACE": "Koncentrace",
}
mapping_en = {
    "DAVKOVY_PRIKON": "Dose Rate",
    "KONCENTRACE": "Concentration",
}

## 3. Pomocné funkce

### 3.1 Práce s výstupními soubory (ArcGIS kompatibilita)

ArcGIS má přísnější požadavky na názvy sloupců než pandas/Excel (délka,
zakázané znaky, diakritika). Funkce `fix_colname_for_arcgis` proto před
uložením každého souboru převede originální názvy sloupců na ArcGIS
kompatibilní tvar; překladový slovník (originál -> ArcGIS název) se
zároveň uloží do YAML souboru ve složce `Y/` jako dokumentace významu
jednotlivých sloupců výsledné vrstvy.


In [ ]:
def fix_colname_for_arcgis(alias):
    """
    Převede daný název sloupce na tvar kompatibilní s ArcGIS
    (bez diakritiky, jednotek v hranatých závorkách a dalších
    nepovolených znaků).
    """
    tags = {
        " [%]": "_pct",
        "[%]": "_pct",
        "(%K)": "_pct_K",
        "*(10) [nSv/hr]": "_10_nSvh",
        "[nGy/h]": "_nGyh",
        " [ppm]": "_ppm",
        " [kBq/m2]": "_kBqm2",
        "Cs-137": "Cs137",
        "[+/-]": "",
        "ř": "r", "ě": "e", "á": "a", "í": "i", "š": "s",
        "č": "c", "ž": "z", "ý": "y", "é": "e", "-": "_",
    }

    name = alias[:64]  # ArcGIS limit na délku názvu sloupce

    for old, new in tags.items():
        name = name.replace(old, new)

    name = name.strip()
    name = re.sub(r"\W|^(?=\d)", "_", name)  # zbylé nepovolené znaky, číslo na začátku

    return name


def save_to_gpkg(srcgdf, fn, layername):
    """
    Uloží `srcgdf` jako vrstvu `layername` do GeoPackage souboru `fn`.
    Názvy sloupců jsou před uložením převedeny do ArcGIS kompatibilního
    tvaru a překladový slovník se uloží do YAML.
    """
    original_to_compatible = {col: fix_colname_for_arcgis(col) for col in srcgdf.columns}
    gdf = srcgdf.rename(columns=original_to_compatible)
    gdf.to_file(fn, layer=layername, driver="GPKG")

    fnb = Path(fn).stem
    Path("Y").mkdir(exist_ok=True)
    with open(f"Y/{fnb}_{layername}_dict.yaml", "w") as file:
        yaml.dump(original_to_compatible, file)
    return gdf


def save_to_geojson(srcgdf, fn, layername):
    """
    Uloží `srcgdf` jako GeoJSON soubor `fn` v souřadnicovém systému EPSG:4326
    (vyžadováno pro ArcGIS Online, RFC 7946). Názvy sloupců jsou před
    uložením převedeny do ArcGIS kompatibilního tvaru a překladový slovník
    se uloží do YAML.
    """
    original_to_compatible = {col: fix_colname_for_arcgis(col) for col in srcgdf.columns}
    gdf = srcgdf.rename(columns=original_to_compatible)
    gdf.to_crs(epsg=4326).to_file(fn, driver="GeoJSON")

    fnb = Path(fn).stem
    Path("Y").mkdir(exist_ok=True)
    with open(f"Y/{fnb}_{layername}_dict.yaml", "w") as file:
        yaml.dump(original_to_compatible, file)
    return gdf


def read_sample_layer_from_gpkg():
    """
    Načte vrstvu 'assays' z 'results.gpkg' a vrátí GeoDataFrame se sloupci
    přejmenovanými zpět na originální (české) názvy podle uloženého YAML
    slovníku.
    """
    assay_pts = gpd.read_file("results.gpkg", layer="assays")
    with open("results_assays_dict.yaml", "r") as file:
        orig_to_compat = yaml.safe_load(file)
        compat_to_orig = {v: k for k, v in orig_to_compat.items()}
    gdf = assay_pts.rename(columns=compat_to_orig)
    return gdf

### 3.2 Rozpoznání typu a lokality vzorku z jeho názvu

Vzorky se v Excelu jmenují podle konvence `<zkratka_typu><číslo_bodu><volitelně A/B>`
(např. `S12`, `LH07B`). Následující funkce z tohoto názvu odvodí:
- `extract_bod` — číslo odběrového bodu,
- `urceni_vzorku` / `urceni_vzorku_en` — český/anglický název typu vzorku
  (u lesního humusu navíc rozliší horizont A/B).


In [ ]:
def extract_bod(vzorek):
    """Vrátí číslo odběrového bodu z názvu vzorku (číslice na konci, případně
    zakončené písmenem A/B u humusu). Pokud název neodpovídá vzoru, vrátí None."""
    match = re.search(r"(\d+)[AB]?$", vzorek)
    if not match:
        return None
    num = match.group(1)
    if len(num) <= 2:
        return int(num)
    return int(num[-2:])  # u delších čísel se použijí poslední dvě číslice


def urceni_vzorku(nazev):
    """Vrátí český název typu vzorku podle prefixu v jeho názvu."""
    if len(nazev) >= 2 and nazev[:2] in typ_vzorku:
        typ = typ_vzorku[nazev[:2]]
    elif nazev[:1] in typ_vzorku:
        typ = typ_vzorku[nazev[:1]]
    else:
        return "Není definován"

    if typ == "Lesní nadložní humus":
        posledni_pismeno = nazev[-1] if nazev[-1].isalpha() else None
        if posledni_pismeno in typ_humusu:
            return f"{typ} {typ_humusu[posledni_pismeno]}"
    return typ


def urceni_vzorku_en(nazev):
    """Anglická obdoba `urceni_vzorku` (pro dvojjazyčné popupy v ArcGIS/StoryMaps)."""
    if len(nazev) >= 2 and nazev[:2] in typ_vzorku_en:
        typ = typ_vzorku_en[nazev[:2]]
    elif nazev[:1] in typ_vzorku_en:
        typ = typ_vzorku_en[nazev[:1]]
    else:
        return "Not defined"

    if typ == "Forest Floor Humus":
        posledni_pismeno = nazev[-1] if nazev[-1].isalpha() else None
        if posledni_pismeno in typ_humusu_en:
            return f"{typ} {typ_humusu_en[posledni_pismeno]}"
    return typ

## 4. Zpracování dat po jednotlivých letech

Pro každý rok se z `ETEDATA.xlsx` načtou tři listy:

| List | Vždy přítomen? | Obsah |
|---|---|---|
| `ETE_<rok>` | ano | laboratorní vzorky (mech, kůra, humus, borůvky, houby...) |
| `ETE_<rok>_In-situ` | ne | in-situ měření kermového příkonu ve vzduchu |
| `ETE_<rok>_GT40` | ne | měření spektrometrem GT-40 |

U všech tří se doplní rok, souřadnice (spojením podle `BOD_ID`) a popisné
sloupce (typ vzorku / měřená veličina / jednotky v CZ i EN).

**Revize bodu 2023-2024:** od roku 2024 je bod historicky vedený pod číslem
29 nahrazen bodem 31. Aby šel spojit se správnými (novými) souřadnicemi,
číslo bodu se dočasně přepíše na 31 už před spojováním (`merge`) a hned po
něm vrátí zpět na 29 -- napříč lety tak zůstává konzistentní identifikátor
`BOD_ID`, ale souřadnice odpovídají aktuální poloze bodu. Při další revizi
bodu stačí přidat analogickou dvojici podmínek (viz zakomentovaný vzor
`ROK_ZMENY` / `PŮVODNÍ_BOD` / `NOVÝ_BOD` v kódu).


In [ ]:
# Pole pro jednotlivé druhy měření (jeden list na rok se sem přidá jako DataFrame)
vsechny_dily = []
vsechny_dily_insitu = []
vsechny_dily_gt = []

for Year in Rok:
    # --- Laboratorní vzorky -------------------------------------------------
    dataframe_samples = pd.read_excel(Soubor, sheet_name=f"ETE_{Year}", decimal=",")

    Year = int(Year)
    dataframe_samples["ROK_ODBERU"] = float(Year)  # float kvůli filtru v ArcGIS dashboardu

    dataframe_samples["BOD_ID"] = dataframe_samples["Název vzorku"].apply(extract_bod)
    if Year >= 2024:
        dataframe_samples["BOD_ID"] = dataframe_samples["BOD_ID"].replace(29, 31)
    """
    else if Year >= ROK_ZMENY:
        dataframe_samples['BOD_ID'] = dataframe_samples['BOD_ID'].replace(PŮVODNÍ_BOD, NOVÝ_BOD)
    """

    dataframe_samples["TYP_VZORKU"] = dataframe_samples["Název vzorku"].apply(urceni_vzorku)
    dataframe_samples["TYP_VZORKU_en"] = dataframe_samples["Název vzorku"].apply(urceni_vzorku_en)

    dataframe_samples["POZNAMKA"] = dataframe_samples["Poznámka"]

    dataframe_samples["ZPUSOB_MERENI"] = "Laboratorní gama spektrometrie"
    dataframe_samples["ZPUSOB_MERENI_en"] = "Laboratory gamma spectrometry"

    dataframe_samples["MERENA_VELICINA"] = "Hmotnostní aktivita"
    dataframe_samples["MERENA_VELICINA_en"] = "Activity per unit mass"

    # Jednotka Bq/m2 odpovídá plošné (ne hmotnostní) aktivitě -- přepíšeme popis veličiny.
    dataframe_samples.loc[dataframe_samples["JEDNOTKA"] == "Bq/m2", "MERENA_VELICINA"] = "Plošná aktivita"
    dataframe_samples.loc[dataframe_samples["JEDNOTKA"] == "Bq/m2", "MERENA_VELICINA_en"] = "Activity per unit area"

    dataframe_samples["HODNOTA"] = pd.to_numeric(dataframe_samples["HODNOTA"], errors="coerce")

    dataframe_samples = dataframe_samples.merge(dataframe_souradnice, on=["BOD_ID"])

    if Year >= 2024:
        dataframe_samples["BOD_ID"] = dataframe_samples["BOD_ID"].replace(31, 29)

    vsechny_dily.append(dataframe_samples)

    # --- In-situ měření (list nemusí pro daný rok existovat) ----------------
    try:
        dataframe_insitu = pd.read_excel(Soubor, sheet_name=f"ETE_{Year}_In-situ", decimal=",")
        ma_insitu = True
    except Exception:
        ma_insitu = False

    if ma_insitu:
        dataframe_insitu["ROK_ODBERU"] = float(Year)

        if Year >= 2024:
            dataframe_insitu["BOD_ID"] = dataframe_insitu["BOD_ID"].replace(29, 31)

        dataframe_insitu["ZPUSOB_MERENI"] = "Měření In-Situ"
        dataframe_insitu["ZPUSOB_MERENI_en"] = "In-Situ Measurements"

        dataframe_insitu["MERENA_VELICINA"] = "Kermový příkon ve vzduchu"
        dataframe_insitu["MERENA_VELICINA_en"] = "Air kerma rate"

        dataframe_insitu["CHYBA_MERENI"] = pd.to_numeric(
            dataframe_insitu["Chyba kermového příkonu ve vzduchu [%]"], errors="coerce"
        )
        dataframe_insitu["JEDNOTKA"] = "nGy/h"
        dataframe_insitu["HODNOTA"] = pd.to_numeric(
            dataframe_insitu["Kermový příkon ve vzduchu [nGy/h]"], errors="coerce"
        )

        dataframe_insitu = dataframe_insitu.merge(dataframe_souradnice, on=["BOD_ID"])

        if Year >= 2024:
            dataframe_insitu["BOD_ID"] = dataframe_insitu["BOD_ID"].replace(31, 29)

        vsechny_dily_insitu.append(dataframe_insitu)

    # --- Měření GT-40 (list nemusí pro daný rok existovat) ------------------
    try:
        dataframe_gt40 = pd.read_excel(Soubor, sheet_name=f"ETE_{Year}_GT40", decimal=",")
    except Exception:
        continue

    dataframe_gt40["ROK_ODBERU"] = float(Year)

    if Year >= 2024:
        dataframe_gt40["BOD_ID"] = dataframe_gt40["BOD_ID"].replace(29, 31)

    dataframe_gt40["ZPUSOB_MERENI"] = "Měření s GT-40"
    dataframe_gt40["ZPUSOB_MERENI_en"] = "GT-40 measurements"

    dataframe_gt40["MERENA_VELICINA_en"] = dataframe_gt40["MERENA_VELICINA"].map(mapping_en)
    dataframe_gt40["MERENA_VELICINA"] = dataframe_gt40["MERENA_VELICINA"].map(mapping_cz)

    dataframe_gt40 = dataframe_gt40.merge(dataframe_souradnice, on=["BOD_ID"], how="left")

    if Year >= 2024:
        dataframe_gt40["BOD_ID"] = dataframe_gt40["BOD_ID"].replace(31, 29)

    vsechny_dily_gt.append(dataframe_gt40)

## 5. Sestavení finálních tabulek a GeoDataFrame

Tři seznamy dílčích ročních tabulek se spojí do tří finálních `DataFrame`,
sloupce se srovnají do jednotného pořadí a přidá se geometrie bodu
(ze souřadnic `LATITUDE`/`LONGITUDE`, systém EPSG:4326).


In [ ]:
df_fin_gt = pd.concat(vsechny_dily_gt, ignore_index=True)
df_reordered_gt = df_fin_gt.loc[
    :, ["NAZEV_BODU", "BOD_ID", "LATITUDE", "LONGITUDE", "ROK_ODBERU",
        "ZPUSOB_MERENI", "ZPUSOB_MERENI_en", "RADIONUKLID",
        "MERENA_VELICINA", "MERENA_VELICINA_en", "HODNOTA", "JEDNOTKA"]
].sort_values(by=["ROK_ODBERU", "BOD_ID"])

df_fin_insitu = pd.concat(vsechny_dily_insitu, ignore_index=True)
df_reordered_insitu = df_fin_insitu.loc[
    :, ["NAZEV_BODU", "BOD_ID", "LATITUDE", "LONGITUDE", "ROK_ODBERU",
        "ZPUSOB_MERENI", "ZPUSOB_MERENI_en", "MERENA_VELICINA",
        "MERENA_VELICINA_en", "HODNOTA", "CHYBA_MERENI", "JEDNOTKA"]
].sort_values(by=["ROK_ODBERU", "BOD_ID"])

df_fin = pd.concat(vsechny_dily, ignore_index=True)
df_reordered = df_fin.loc[
    :, ["NAZEV_BODU", "BOD_ID", "LATITUDE", "LONGITUDE", "TYP_VZORKU",
        "TYP_VZORKU_en", "ROK_ODBERU", "ZPUSOB_MERENI", "ZPUSOB_MERENI_en",
        "RADIONUKLID", "MERENA_VELICINA", "MERENA_VELICINA_en", "HODNOTA",
        "JEDNOTKA", "POZNAMKA"]
].sort_values(by=["ROK_ODBERU", "BOD_ID"])

# Geometrie bodů
geometry = gpd.points_from_xy(df_reordered["LONGITUDE"], df_reordered["LATITUDE"], crs="EPSG:4326")
geometry_insitu = gpd.points_from_xy(df_reordered_insitu["LONGITUDE"], df_reordered_insitu["LATITUDE"], crs="EPSG:4326")
geometry_gt = gpd.points_from_xy(df_reordered_gt["LONGITUDE"], df_reordered_gt["LATITUDE"], crs="EPSG:4326")

gdf = gpd.GeoDataFrame(df_reordered, geometry=geometry)
gdf_insitu = gpd.GeoDataFrame(df_reordered_insitu, geometry=geometry_insitu)
gdf_gt = gpd.GeoDataFrame(df_reordered_gt, geometry=geometry_gt)

## 6. Export do GeoJSON

GeoPackage (`.gpkg`) je vhodný pro lokální/QGIS zpracování (viz `save_to_gpkg`
výše), pro doplňování dat v ArcGIS Online se však nehodí — proto se ukládá
přímo GeoJSON. Výstupem jsou tři samostatné vrstvy.


In [ ]:
Path("OUTPUT").mkdir(exist_ok=True)

save_to_geojson(gdf, Path("OUTPUT") / "database_samples.geojson", "database")
save_to_geojson(gdf_insitu, Path("OUTPUT") / "database_insitu.geojson", "database_insitu")
save_to_geojson(gdf_gt, Path("OUTPUT") / "database_gt40.geojson", "database_gt40")

print("HOTOVO")

## 7. Kontrola výstupů

Sanity-check po zpracování — ověření, že se všechny typy vzorků správně
rozpoznaly a že vybraný bod má očekávaný počet řádků.


In [ ]:
df_reordered["TYP_VZORKU_en"].unique()

In [ ]:
df_reordered.explode("BOD_ID")

## 8. Časový vývoj $^{137}$Cs podle typu vzorku

Pro každý typ vzorku (mech, kůra, humus A/B, borůvky, houby...) je ukázán
časový vývoj aktivity $^{137}$Cs a posouzeno, zda pokles odpovídá čistě
fyzikální přeměně, nebo je rychlejší (což ukazuje na vliv vnějších faktorů
— vyplavování z povrchu, migraci do hlubších vrstev půdního profilu apod.).

**Popisná část:** pro každý rok a typ vzorku je spočítán medián a k němu
25.—75. percentil (užší pásmo) a min-max (širší pásmo).

**Kvantitativní část:** na mediány je metodou nejmenších čtverců na
zlogaritmovaných hodnotách proložen:
- **empirický exponenciální fit** — efektivní poločas $T_{ef}$, se kterým
  data skutečně klesají,
- **čistě fyzikální rozpad** — referenční přímka se sklonem daným
  fyzikálním poločasem $^{137}$Cs, $T_{fyz} = 30{,}018$ let, ukotvená
  v první dostupné hodnotě.

$$\frac{1}{T_{ef}} = \frac{1}{T_{fyz}} + \frac{1}{T_{vnejsi}}$$

Roky ovlivněné kůrovcovou kalamitou (2015-2020) a revizí bodů (2023-24) jsou
u humusu v grafu vyznačené, ale z fitu vyloučené (`EXCLUDE_YEARS_BY_TYPE`).


In [ ]:
# --- Nastavení analýzy -------------------------------------------------------

MIN_YEARS = 5      # minimální počet let s daty, aby fit dával smysl
T_PHYS = 30.018    # let; fyzikální poločas přeměny Cs-137 (LNHB, http://www.lnhb.fr/nuclides/Cs-137_tables.pdf)

# Roky vyloučené z fitu (ne z grafu) pro konkrétní typy vzorku -- zde kvůli
# kůrovcové kalamitě a souvisejícím revizím odběrových bodů
EXCLUDE_YEARS_BY_TYPE = {
    "Lesní nadložní humus nehumifikovaný (litter)": set(range(2015, 2021)),
    "Lesní nadložní humus humifikovaný (fermenton)": set(range(2015, 2021)),
    "Lesní nadložní humus": set(range(2015, 2021)),
}

In [ ]:
# --- Příprava dat -------------------------------------------------------------

# Analýza pracuje jen s Cs-137 (sloupec HODNOTA obsahuje i K-40, Ra-226, Th-232)
df_cs = df_reordered[df_reordered["RADIONUKLID"] == "Cs-137"].dropna(subset=["HODNOTA"]).copy()
df_cs["ROK_ODBERU"] = df_cs["ROK_ODBERU"].astype(int)

# Medián, kvartily a min/max po letech a typu vzorku
agg = (
    df_cs.groupby(["TYP_VZORKU", "ROK_ODBERU"])["HODNOTA"]
    .agg(
        median="median",
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
        min="min",
        max="max",
        n="count",
    )
    .reset_index()
    .sort_values(["TYP_VZORKU", "ROK_ODBERU"])
)

In [ ]:
# --- Fit efektivního poločasu -------------------------------------------------

def fit_effective_halflife(years, medians):
    years = np.asarray(years, dtype=float)
    y = np.log(np.asarray(medians, dtype=float))
    t0 = years.min()

    res = stats.linregress(years - t0, y)
    lam_eff = -res.slope
    if lam_eff <= 0:
        return None

    T_eff = np.log(2) / lam_eff
    T_eff_stderr = abs(T_eff * (res.stderr / res.slope))  # delta metoda

    return {
        "T_eff": T_eff,
        "T_eff_stderr": T_eff_stderr,
        "r2": res.rvalue ** 2,
        "slope": res.slope,
        "intercept": res.intercept,
        "t0": t0,
        "n_points": len(years),
    }


def dopocti_vliv_vnejsich_faktoru(T_eff, T_phys=T_PHYS):

    if T_eff >= T_phys:
        return np.inf
    return 1.0 / (1.0 / T_eff - 1.0 / T_phys)

In [ ]:
# --- Grafy a fity pro každý typ vzorku ----------------------------------------

results = []

for typ, sub in agg.groupby("TYP_VZORKU"):
    sub = sub.sort_values("ROK_ODBERU")
    if len(sub) < MIN_YEARS:
        continue

    jednotka = df_cs.loc[df_cs["TYP_VZORKU"] == typ, "JEDNOTKA"].mode().iat[0]
    velicina = df_cs.loc[df_cs["TYP_VZORKU"] == typ, "MERENA_VELICINA"].mode().iat[0]

    excluded = EXCLUDE_YEARS_BY_TYPE.get(typ, set())
    sub_fit = sub[~sub["ROK_ODBERU"].isin(excluded)]

    fit = None
    if len(sub_fit) >= MIN_YEARS:
        fit = fit_effective_halflife(sub_fit["ROK_ODBERU"], sub_fit["median"])

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.fill_between(sub["ROK_ODBERU"], sub["min"], sub["max"],
                     alpha=0.15, color="tab:blue", label="min-max")
    ax.fill_between(sub["ROK_ODBERU"], sub["p25"], sub["p75"],
                     alpha=0.35, color="tab:blue", label="25.-75. percentil")
    ax.plot(sub["ROK_ODBERU"], sub["median"], "o-", color="tab:blue", label="medián")

    for rok_vyl in sorted(excluded):
        ax.axvspan(rok_vyl - 0.5, rok_vyl + 0.5, color="gray", alpha=0.15)

    years_line = np.array([sub["ROK_ODBERU"].min(), sub["ROK_ODBERU"].max()], dtype=float)

    A0, t0_phys = sub["median"].iloc[0], sub["ROK_ODBERU"].iloc[0]
    phys_line = A0 * np.exp(-np.log(2) / T_PHYS * (years_line - t0_phys))
    ax.plot(years_line, phys_line, ":", color="black",
             label=f"čistý radioaktivní rozpad (T={T_PHYS:.3f} let)")

    if fit:
        fit_line = np.exp(fit["intercept"] + fit["slope"] * (years_line - fit["t0"]))
        ax.plot(years_line, fit_line, "--", color="tab:red",
                 label=f"empirický fit (T_ef={fit['T_eff']:.1f} let, R²={fit['r2']:.2f})")

    ax.set_yscale("log")
    ax.set_xlabel("Rok odběru")
    ax.set_ylabel(f"{velicina} $^{{137}}$Cs [{jednotka}]")
    ax.set_title(typ)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.savefig(f'{typ}.png', dpi=800)
    plt.show()

    results.append({
        "typ_vzorku": typ,
        "velicina": velicina,
        "jednotka": jednotka,
        "n_let_celkem": len(sub),
        "n_let_ve_fitu": len(sub_fit),
        "vyloucene_roky": sorted(excluded) if excluded else None,
        "T_ef_let": fit["T_eff"] if fit else np.nan,
        "T_ef_stderr_let": fit["T_eff_stderr"] if fit else np.nan,
        "R2": fit["r2"] if fit else np.nan,
        "T_fyz_let": T_PHYS,
        "T_vnejsi_let": dopocti_vliv_vnejsich_faktoru(fit["T_eff"]) if fit else np.nan,
    })

df_vysledky = pd.DataFrame(results)
df_vysledky

## 9. Prostorová interpolace (IDW) mediánu aktivity $^{137}$Cs v mechu v čase


In [ ]:
# --- Prostorová interpolace (IDW) mediánu aktivity Cs-137 v čase ------------

from scipy.spatial.distance import cdist
from matplotlib.colors import LogNorm

# Souřadnice středu kontejnmentu 1. bloku JE Temelín (WGS84)
ETE_LAT, ETE_LON = 49.1798801575377, 14.377222930080388

CRS_PROJ = "EPSG:5514"    # S-JTSK / Křovák — souřadnice v metrech, vhodné pro ČR
RADIUS = 25               # poloměr zájmové oblasti kolem ETE [km]
ZONES = [5, 13]           # vyznačené havarijní zóny [km]
GRID_N = 200              # rozlišení interpolační mřížky (GRID_N x GRID_N bodů)
IDW_POWER = 2             # mocninný parametr IDW (běžná volba)
VMAX = 5_000              # pevný horní konec barevné škály (stejný napříč typy i roky)
YEAR = [2000, 2010, 2020, 2024]
YEAR_HUMUS = [2001, 2010, 2020, 2024]   # humus se odebíral až od roku 2001

# Souřadnice ETE v projektovaném systému (společné pro všechny typy vzorku)
gdf_ete = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy([ETE_LON], [ETE_LAT]), crs="EPSG:4326"
).to_crs(CRS_PROJ)
ete_x, ete_y = gdf_ete.geometry.x.iloc[0], gdf_ete.geometry.y.iloc[0]

# Pravidelná mřížka bodů v čtvercové oblasti kolem ETE (společná pro všechny typy)
xs = np.linspace(ete_x - RADIUS * 1000, ete_x + RADIUS * 1000, GRID_N)
ys = np.linspace(ete_y - RADIUS * 1000, ete_y + RADIUS * 1000, GRID_N)
grid_x, grid_y = np.meshgrid(xs, ys)
grid_xy = np.column_stack([grid_x.ravel(), grid_y.ravel()])

dist_from_ete = np.hypot(grid_xy[:, 0] - ete_x, grid_xy[:, 1] - ete_y)
mask_outside = dist_from_ete > RADIUS * 1000

extent_km = [(xs.min() - ete_x) / 1000, (xs.max() - ete_x) / 1000,
             (ys.min() - ete_y) / 1000, (ys.max() - ete_y) / 1000]


def idw_interpolate(sample_xy, sample_values, grid_xy, power=IDW_POWER):
    """
    Interpolace metodou IDW (Inverse Distance Weighting). Váha každého
    vzorku je nepřímo úměrná jeho vzdálenosti od interpolovaného bodu
    umocněné na `power`. Bod ležící přesně na místě vzorku dostane váhu
    tohoto vzorku (ošetřeno nahrazením nulové vzdálenosti malým číslem).
    """
    d = cdist(grid_xy, sample_xy)
    d[d == 0] = 1e-6
    w = 1.0 / d ** power
    return (w @ sample_values) / w.sum(axis=1)


def plot_idw_mapy(typ_vzorku, years=YEAR):
    """
    Vykreslí 2x2 mřížku map IDW interpolace mediánu aktivity Cs-137 pro
    daný typ vzorku, pro roky uvedené v `years`. Barevná škála je
    logaritmická s pevným horním koncem (VMAX), stejným pro všechny typy
    vzorku i roky, aby byly mapy mezi sebou vizuálně srovnatelné. Roky bez
    dostatku dat (< 3 vzorky) se zobrazí jako prázdný panel s poznámkou.
    """
    sub_typ = df_cs[df_cs["TYP_VZORKU"] == typ_vzorku]
    jednotka = sub_typ["JEDNOTKA"].mode().iat[0]
    velicina = sub_typ["MERENA_VELICINA"].mode().iat[0]

    interpolated = {}
    sample_points = {}
    vmin_data = np.inf

    for year in years:
        sub = sub_typ[sub_typ["ROK_ODBERU"] == year].dropna(subset=["HODNOTA"])
        if len(sub) < 3:
            interpolated[year] = None
            continue

        gdf_pts = gpd.GeoDataFrame(
            sub, geometry=gpd.points_from_xy(sub["LONGITUDE"], sub["LATITUDE"]), crs="EPSG:4326"
        ).to_crs(CRS_PROJ)

        sample_xy = np.column_stack([gdf_pts.geometry.x, gdf_pts.geometry.y])
        log_hodnoty = np.log(sub["HODNOTA"].values)

        interp = np.exp(idw_interpolate(sample_xy, log_hodnoty, grid_xy))
        interp[mask_outside] = np.nan

        interpolated[year] = interp.reshape(GRID_N, GRID_N)
        sample_points[year] = sample_xy
        vmin_data = min(vmin_data, np.nanmin(interp))

    fig, axes = plt.subplots(2, 2, figsize=(11, 11))
    axes = axes.ravel()

    im = None
    for ax, year in zip(axes, years):
        if interpolated[year] is None:
            ax.text(0.5, 0.5, "Nedostatek dat", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(str(year))
            ax.set_xlim(extent_km[0], extent_km[1])
            ax.set_ylim(extent_km[2], extent_km[3])
            ax.set_aspect("equal")
            continue

        im = ax.imshow(
            interpolated[year], extent=extent_km, origin="lower",
            norm=LogNorm(vmin=vmin_data, vmax=VMAX), cmap="viridis",
        )

        sx, sy = sample_points[year][:, 0], sample_points[year][:, 1]
        ax.scatter((sx - ete_x) / 1000, (sy - ete_y) / 1000, s=12, color="white", edgecolor="black", linewidth=0.5)

        for r in ZONES:
            ax.add_patch(plt.Circle((0, 0), r, fill=False, color="red", linestyle="--", linewidth=1))
        ax.plot(0, 0, marker="*", color="red", markersize=14, markeredgecolor="black")

        ax.set_title(str(year))
        ax.set_xlabel("Vzdálenost od ETE [km]")
        ax.set_ylabel("Vzdálenost od ETE [km]")
        ax.set_aspect("equal")

    if im is not None:
        fig.colorbar(im, ax=axes.tolist(), shrink=0.8, label=f"{velicina} $^{{137}}$Cs [{jednotka}] – {typ_vzorku}")
    fig.suptitle(f"Prostorové rozložení aktivity $^{{137}}$Cs – {typ_vzorku} (IDW interpolace)")

    out_name = re.sub(r"\W+", "_", typ_vzorku).strip("_").lower()
    plt.savefig(Path("OUTPUT") / f"cs137_idw_{out_name}.png", dpi=600, bbox_inches="tight")
    plt.show()


# --- Vykreslení pro jednotlivé typy vzorku ----------------------------------
plot_idw_mapy("Mech trávník Schreberův (Pleurozium schreberi)")
plot_idw_mapy("Vnější kůra borovice lesní (Pinus sylvestris)")
plot_idw_mapy("Lesní nadložní humus humifikovaný (fermenton)", years=YEAR_HUMUS)